In [1]:
# Cell 1: Environment Setup
import os
import sys

# Check if running on Kaggle
IS_KAGGLE = os.path.exists('/kaggle')
print(f"Running on Kaggle: {IS_KAGGLE}")

# Set directories
if IS_KAGGLE:
    WORKING_DIR = '/kaggle/working'
    RESULTS_DIR = '/kaggle/working/results'
    # Add repository to Python path
    sys.path.insert(0, '/kaggle/input/gdsearch-repository')
else:
    # Running locally
    WORKING_DIR = os.path.abspath('..')
    RESULTS_DIR = os.path.join(WORKING_DIR, 'results')
    # Add parent directory to Python path
    sys.path.insert(0, WORKING_DIR)

os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Working directory: {WORKING_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Python path updated: {sys.path[0]}")

Running on Kaggle: False
Working directory: /workspaces/GDSearch
Results directory: /workspaces/GDSearch/results
Python path updated: /workspaces/GDSearch


In [2]:
# Cell 2: Install Dependencies (Kaggle Only)
if IS_KAGGLE:
    print("📦 Installing dependencies...")
    
    # Set critical environment variables FIRST to avoid warnings
    import os
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # Prevent tokenizer fork warnings
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
    os.environ['HF_HUB_DISABLE_XET'] = '1'
    os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
    os.environ['HF_HUB_OFFLINE'] = '0'
    # Suppress CUDA warnings
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    os.environ['GRPC_VERBOSITY'] = 'ERROR'
    os.environ['GLOG_minloglevel'] = '2'
    
    # Upgrade core packages first to avoid compatibility issues
    !pip install -q --upgrade pip setuptools wheel
    
    # FIXED: Install packages with exact compatible versions to avoid conflicts
    # The order matters - install conflicting packages first with specific versions
    !pip install -q --upgrade "fsspec==2025.3.0"  # FIXED: Exact version for gcsfs
    !pip install -q --upgrade "pyarrow>=14.0.0,<20.0.0"  # FIXED: Compatible with cudf-cu12
    !pip install -q --upgrade "rich>=12.4.4,<14"  # FIXED: Compatible with bigframes
    !pip install -q --upgrade "click>=7.0,!=8.3.0"  # FIXED: Compatible with ray
    !pip install -q --upgrade "cryptography>=19.0,<44"  # FIXED: Compatible with pydrive2
    !pip install -q --upgrade "pyOpenSSL>=19.1.0,<=24.2.1"  # FIXED: Compatible with pydrive2
    !pip install -q --upgrade "huggingface_hub>=0.30.0,<1.0"
    !pip install -q --upgrade "protobuf>=3.20.3,<4.0.0"  # FIXED: Compatible with TensorFlow
    
    # Install remaining required packages
    !pip install -q transformers datasets plotly kaleido psutil scipy optuna mlflow
    
    print("✅ All dependencies installed!")
    
    # Verify datasets loading works
    print("\n🔍 Verifying HuggingFace compatibility...")
    try:
        from datasets import load_dataset
        from transformers import AutoTokenizer
        import transformers
        
        # Suppress unnecessary warnings
        transformers.logging.set_verbosity_error()
        import warnings
        warnings.filterwarnings('ignore', message='Some weights.*were not initialized')
        warnings.filterwarnings('ignore', message='.*cuFFT.*')
        warnings.filterwarnings('ignore', message='.*cuDNN.*')
        warnings.filterwarnings('ignore', message='.*cuBLAS.*')
        
        # Quick test
        _ = AutoTokenizer.from_pretrained('distilbert-base-uncased')
        print("✅ HuggingFace models accessible")
    except Exception as e:
        print(f"⚠️  HuggingFace access limited: {e}")
        print("   NLP experiments will use fallback mode (simpler models)")
        print("   This is normal for some Kaggle environments.")
else:
    print("⚠️  Not on Kaggle - assuming dependencies are already installed")
    # Set environment variables for local runs too
    import os
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

⚠️  Not on Kaggle - assuming dependencies are already installed


In [3]:
# Cell 2.5: Restore Previous Checkpoints (Resume Logic)
# PHASE 6.2 FIX: Copy checkpoints from persistent storage

import shutil

checkpoint_input_dir = '/kaggle/input/gdsearch-checkpoints/checkpoints'
checkpoint_working_dir = '/kaggle/working/results/checkpoints'

if IS_KAGGLE and os.path.exists(checkpoint_input_dir):
    print("📦 Restoring previous checkpoints from persistent storage...")
    
    try:
        # Create working checkpoint directory
        os.makedirs(checkpoint_working_dir, exist_ok=True)
        
        # Count available checkpoints
        checkpoint_files = [f for f in os.listdir(checkpoint_input_dir) 
                          if f.endswith('.pt') or f.endswith('.pth')]
        
        if checkpoint_files:
            # Copy all checkpoint files
            for ckpt_file in checkpoint_files:
                src = os.path.join(checkpoint_input_dir, ckpt_file)
                dst = os.path.join(checkpoint_working_dir, ckpt_file)
                shutil.copy2(src, dst)
            
            print(f"✅ Restored {len(checkpoint_files)} checkpoint files")
            print(f"   Source: {checkpoint_input_dir}")
            print(f"   Destination: {checkpoint_working_dir}")
            print("\n💡 You can now use --resume flag to continue training")
        else:
            print("⚠️  No checkpoint files found in input dataset")
    except Exception as e:
        print(f"⚠️  Failed to restore checkpoints: {e}")
        print("   This is non-critical. Training will start from scratch.")
else:
    if IS_KAGGLE:
        print("ℹ️  No previous checkpoints found - starting fresh")
        print(f"   (Checked: {checkpoint_input_dir})")
    else:
        print("ℹ️  Not on Kaggle - checkpoint restoration skipped")

print("")

ℹ️  Not on Kaggle - checkpoint restoration skipped



In [4]:
# Cell 3: GPU & Environment Check
import torch

print("=" * 60)
print("GPU Configuration")
print("=" * 60)

if torch.cuda.is_available():
    print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
    GPU_FLAG = "--kaggle-t4"
else:
    print("⚠️ No GPU available - training will be slower")
    GPU_FLAG = ""

print(f"\nPyTorch: {torch.__version__}")
print(f"Python: {sys.version}")

GPU Configuration
⚠️ No GPU available - training will be slower

PyTorch: 2.9.0+cpu
Python: 3.12.1 (main, Jul 10 2025, 11:57:50) [GCC 13.3.0]


In [ ]:
# Cell 4.5: Quick Validation Test (OPTIONAL - Run this to verify bug fixes)
# This cell runs a quick 2-minute test to ensure all critical bugs are fixed
# You can skip this if you're confident and want to run the full suite directly

print("="*70)
print("🧪 RUNNING QUICK VALIDATION TEST")
print("="*70)
print("This will test:") 
print("  ✓ Training loop indentation fix (should show acc > 85% in epoch 1)")
print("  ✓ Division by zero protection")
print("  ✓ Sanity checks")
print("  ✓ Metric calculations")
print("\nEstimated time: 2-3 minutes")
print("="*70)

import subprocess
import sys

# Determine script path
if IS_KAGGLE:
    script_path = "/kaggle/input/gdsearch-repository/run_all_kaggle.py"
else:
    script_path = os.path.join(WORKING_DIR, "run_all_kaggle.py")

# Quick validation command
cmd = [
    sys.executable, script_path,
    "--ultra-quick",
    "--seeds", "42",
    "--experiments", "mnist",
    "--results-dir", RESULTS_DIR
]

print(f"\nRunning: {' '.join(cmd[-6:])}\n")
print("="*70 + "\n")

result = subprocess.run(cmd, capture_output=True, text=True)

# Show last 50 lines of output
if result.stdout:
    lines = result.stdout.strip().split('\n')
    print("\n".join(lines[-50:]))

if result.returncode == 0:
    print("\n" + "="*70)
    print("✅ VALIDATION TEST PASSED!")
    print("="*70)
    print("All critical bugs are fixed. Safe to proceed with full run.")
    print("="*70)
else:
    print("\n" + "="*70)
    print("⚠️  VALIDATION TEST HAD WARNINGS")
    print("="*70)
    print("Check the output above. Some warnings are OK.")
    print("As long as you see correct accuracy (>85% in epoch 1), proceed.")
    print("="*70)

🧪 RUNNING QUICK VALIDATION TEST
This will test:
  ✓ Training loop indentation fix (should show acc > 85% in epoch 1)
  ✓ Division by zero protection
  ✓ Sanity checks
  ✓ Metric calculations

Estimated time: 2-3 minutes

Running: --seeds 42 --experiments mnist --results-dir /workspaces/GDSearch/results




In [ ]:
# Cell 5: Run Complete Benchmark Suite
#
# This executes the full production benchmark with:
# ✅ 10 seeds for high statistical power
# ✅ Resume logic (safe to interrupt)
# ✅ Cross-experiment aggregation
# ✅ Statistical analysis (t-tests, effect sizes, power analysis)
# ✅ Hyperparameter tuning with Optuna
# ✅ Interactive visualizations
# ✅ Publication-ready reports
# ✅ Automatic fallback for failed experiments (continues with remaining)
# ✅ Checkpoint restore from persistent Input Dataset (Kaggle ephemeral fix)
#

import subprocess
import time
import shutil
import sys
from pathlib import Path

# =============================================================================
# KAGGLE EPHEMERAL STORAGE FIX: Restore checkpoints from persistent input
# =============================================================================
# On Kaggle, /kaggle/working is ephemeral and gets wiped on session restart.
# To resume training, we store checkpoints in an Input Dataset.
# This section copies checkpoints from the input dataset to /kaggle/working.

PERSISTENT_CHECKPOINT_SOURCE = Path("/kaggle/input/gdsearch-checkpoints")
CHECKPOINT_DEST = Path(RESULTS_DIR) / "checkpoints"

def restore_checkpoints_from_input():
    """Restore checkpoints from persistent Input Dataset to ephemeral working dir."""
    if not IS_KAGGLE:
        print("   ⏭️  Not on Kaggle - skipping checkpoint restore")
        return 0
    
    if not PERSISTENT_CHECKPOINT_SOURCE.exists():
        print("   ℹ️  No persistent checkpoint dataset found at:")
        print(f"      {PERSISTENT_CHECKPOINT_SOURCE}")
        print("   💡 To enable resume across sessions:")
        print("      1. Create a new Dataset named 'gdsearch-checkpoints'")
        print("      2. After each run, upload results/checkpoints/ to it")
        print("      3. Add the dataset as an input to this notebook")
        return 0
    
    # Count checkpoints to restore
    checkpoint_files = list(PERSISTENT_CHECKPOINT_SOURCE.glob("**/*.pt"))
    checkpoint_files += list(PERSISTENT_CHECKPOINT_SOURCE.glob("**/*.pth"))
    checkpoint_files += list(PERSISTENT_CHECKPOINT_SOURCE.glob("**/*.ckpt"))
    
    if not checkpoint_files:
        print("   ℹ️  Persistent checkpoint dataset exists but contains no checkpoints")
        return 0
    
    # Create destination and copy
    CHECKPOINT_DEST.mkdir(parents=True, exist_ok=True)
    
    restored = 0
    for src_file in checkpoint_files:
        # Preserve directory structure relative to source
        relative_path = src_file.relative_to(PERSISTENT_CHECKPOINT_SOURCE)
        dest_file = CHECKPOINT_DEST / relative_path
        dest_file.parent.mkdir(parents=True, exist_ok=True)
        
        # Only copy if destination doesn't exist or is older
        if not dest_file.exists():
            shutil.copy2(src_file, dest_file)
            restored += 1
            print(f"   ✓ Restored: {relative_path}")
    
    return restored

print("="*70)
print("📦 RESTORING CHECKPOINTS FROM PERSISTENT STORAGE")
print("="*70)
try:
    restored_count = restore_checkpoints_from_input()
    if restored_count > 0:
        print(f"\n   ✅ Restored {restored_count} checkpoint files")
        print(f"   Resume will continue from last saved state")
    else:
        print("\n   ℹ️  No checkpoints to restore - starting fresh")
except Exception as e:
    print(f"\n   ⚠️  Checkpoint restore failed: {e}")
    print("   Continuing without restored checkpoints...")
print("="*70 + "\n")

# =============================================================================
# DETERMINE SCRIPT PATH AND BUILD COMMAND
# =============================================================================
if IS_KAGGLE:
    # On Kaggle, the repository is in the input dataset
    script_path = "/kaggle/input/gdsearch-repository/run_all_kaggle.py"
else:
    # Running locally
    script_path = os.path.join(WORKING_DIR, "run_all_kaggle.py")

# Build command
cmd = [
    sys.executable, script_path,  # Use sys.executable for correct Python
    "--experiments", "all",
    "--seeds", "42,123,456,789,1011,1213,1415,1617,1819,2021",
    "--results-dir", RESULTS_DIR,
    "--resume",
    "--profile"
]

# Add Kaggle T4 optimizations if GPU available
if torch.cuda.is_available():
    cmd.append("--kaggle-t4")

print("="*70)
print("🚀 STARTING GDSEARCH BENCHMARK SUITE")
print("="*70)
print(f"\nCommand: {' '.join(cmd)}\n")
print("This will take several hours. The process is resumable - safe to interrupt.")
print("Individual experiment failures will NOT crash the entire pipeline.")
print("="*70 + "\n")

start_time = time.time()

# Execute the benchmark with proper error capture
# NOTE: We DON'T use check=True because we want to capture partial results even on failure
# Instead, we capture stderr and print it if there's an error
result = subprocess.run(cmd, capture_output=True, text=True)

# Print stdout (real-time output already shown, this is for logs)
if result.stdout:
    print(result.stdout)

elapsed = time.time() - start_time
hours = elapsed / 3600
minutes = (elapsed % 3600) / 60

print("\n" + "="*70)
if result.returncode == 0:
    print("✅ BENCHMARK SUITE COMPLETED SUCCESSFULLY")
else:
    print("⚠️  BENCHMARK SUITE COMPLETED WITH SOME ERRORS")
    print("   (Some experiments may have failed but results are still available)")
    # CRITICAL: Print stderr so bugs aren't hidden
    if result.stderr:
        print("\n📋 ERROR LOG (for debugging):")
        print("-" * 50)
        # Print last 50 lines of stderr to avoid overwhelming output
        stderr_lines = result.stderr.strip().split('\n')
        if len(stderr_lines) > 50:
            print(f"   ... ({len(stderr_lines) - 50} earlier lines omitted)")
        for line in stderr_lines[-50:]:
            print(f"   {line}")
        print("-" * 50)
print("="*70)
print(f"⏱️  Total time: {hours:.1f} hours ({minutes:.1f} minutes)")
print("="*70)

# =============================================================================
# POST-RUN: CHECKPOINT BACKUP INSTRUCTIONS
# =============================================================================
print("\n" + "="*70)
print("💾 CHECKPOINT BACKUP INSTRUCTIONS")
print("="*70)
print("To enable resume across Kaggle sessions:")
print("1. Go to 'Output' tab and download the 'checkpoints' folder")
print("2. Create/update a Dataset named 'gdsearch-checkpoints'")
print("3. Upload the checkpoints folder to that dataset")
print("4. Add 'gdsearch-checkpoints' as an Input to this notebook")
print("5. Re-run this notebook - it will automatically restore checkpoints")
print("="*70)

In [ ]:
# Cell 6: Display Results Summary
import pandas as pd
from pathlib import Path

results_path = Path(RESULTS_DIR)

print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)

# Check if results directory exists
if not results_path.exists():
    print("\n⚠️  Results directory not found. Run Cell 5 first.")
else:
    # List all result files
    csv_files = list(results_path.rglob("*.csv"))
    print(f"\n📁 Found {len(csv_files)} result files:\n")
    
    if len(csv_files) == 0:
        print("   No CSV files found yet. Benchmark may still be running.")
    else:
        for f in sorted(csv_files)[:20]:  # Show first 20
            try:
                print(f"  {f.relative_to(results_path)}")
            except ValueError:
                print(f"  {f}")
        
        if len(csv_files) > 20:
            print(f"  ... and {len(csv_files) - 20} more")
    
    # Show cross-experiment aggregation if exists
    agg_file = results_path / "analysis" / "cross_experiment_aggregation.csv"
    if agg_file.exists():
        print("\n📊 Cross-Experiment Aggregation:")
        try:
            agg_df = pd.read_csv(agg_file)
            display(agg_df)
        except Exception as e:
            print(f"   Error reading aggregation file: {e}")
    
    # Show optimizer rankings if exists
    rank_file = results_path / "analysis" / "optimizer_rankings.csv"
    if rank_file.exists():
        print("\n🏆 Optimizer Rankings:")
        try:
            rank_df = pd.read_csv(rank_file)
            display(rank_df)
        except Exception as e:
            print(f"   Error reading rankings file: {e}")

In [ ]:
# Cell 7: Show Experiment-Specific Results

experiments_dir = results_path / "experiments"

if not experiments_dir.exists():
    print("⚠️  Experiments directory not found yet.")
else:
    found_experiments = False
    for exp_dir in sorted(experiments_dir.iterdir()):
        if exp_dir.is_dir():
            csv_files = list(exp_dir.glob("*.csv"))
            if csv_files:
                found_experiments = True
                print(f"\n{'='*60}")
                print(f"📁 {exp_dir.name.upper()} RESULTS")
                print(f"{'='*60}")
                
                # Try to load and display main results
                for csv_file in sorted(csv_files)[:3]:  # Show first 3
                    try:
                        df = pd.read_csv(csv_file)
                        print(f"\n📄 {csv_file.name}")
                        print(f"   Shape: {df.shape}")
                        if 'optimizer' in df.columns or 'Optimizer' in df.columns:
                            opt_col = 'optimizer' if 'optimizer' in df.columns else 'Optimizer'
                            print(f"   Optimizers: {df[opt_col].unique().tolist()}")
                        display(df.head())
                    except Exception as e:
                        print(f"   Error reading {csv_file.name}: {e}")
    
    if not found_experiments:
        print("   No experiment results found yet.")

In [ ]:
# Cell 8: Visualizations
import matplotlib.pyplot as plt

viz_dir = results_path / "visualizations"

if not viz_dir.exists():
    print("⚠️  Visualizations directory not found yet.")
else:
    # List available visualizations
    html_files = list(viz_dir.rglob("*.html"))
    png_files = list(viz_dir.rglob("*.png"))
    
    print(f"\n📈 Found {len(html_files)} interactive HTML plots")
    print(f"📊 Found {len(png_files)} static PNG plots")
    
    if len(png_files) > 0:
        # Display some PNG plots
        for png_file in sorted(png_files)[:6]:  # Show first 6
            try:
                img = plt.imread(str(png_file))
                fig, ax = plt.subplots(figsize=(10, 6))
                ax.imshow(img)
                ax.axis('off')
                ax.set_title(png_file.stem, fontsize=12)
                plt.tight_layout()
                plt.show()
            except Exception as e:
                print(f"Could not display {png_file.name}: {e}")
    else:
        print("   No PNG visualizations found yet.")
    
    if len(html_files) > 0:
        print(f"\n   Interactive HTML plots saved to: {viz_dir}/interactive/")
        print("   (Download and open in browser to view)")

In [ ]:
# Cell 9: Statistical Analysis Summary

analysis_dir = results_path / "analysis"

if not analysis_dir.exists():
    print("⚠️  Analysis directory not found yet.")
else:
    print("=" * 60)
    print("STATISTICAL ANALYSIS")
    print("=" * 60)
    
    # Show cross-experiment statistics
    stats_file = analysis_dir / "cross_experiment_statistics.csv"
    if stats_file.exists():
        print("\n🔬 Cross-Experiment Statistical Comparisons:")
        try:
            stats_df = pd.read_csv(stats_file)
            display(stats_df)
        except Exception as e:
            print(f"   Error reading statistics file: {e}")
    else:
        print("\n   Cross-experiment statistics not generated yet.")
    
    # Show basic statistics
    basic_stats = analysis_dir / "00_basic_statistics.csv"
    if basic_stats.exists():
        print("\n📊 Basic Statistics:")
        try:
            basic_df = pd.read_csv(basic_stats)
            display(basic_df.head(20))
        except Exception as e:
            print(f"   Error reading basic statistics: {e}")
    else:
        print("\n   Basic statistics not generated yet.")

In [ ]:
# Cell 10: Archive Results for Download
import shutil
from datetime import datetime

if IS_KAGGLE:
    try:
        archive_name = f"gdsearch_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        archive_path = f"/kaggle/working/{archive_name}"
        
        print(f"Creating archive: {archive_name}.zip")
        
        # Check if results directory has content
        if results_path.exists() and any(results_path.iterdir()):
            shutil.make_archive(archive_path, 'zip', RESULTS_DIR)
            print(f"\n✅ Results archived to: {archive_path}.zip")
            print("   Download from Kaggle Output tab")
        else:
            print("\n⚠️  No results to archive yet. Run Cell 5 first.")
    except Exception as e:
        print(f"\n❌ Error creating archive: {e}")
else:
    print(f"Not on Kaggle - results saved to: {RESULTS_DIR}")

In [ ]:
# Cell 12: Quick Access Guide

print("""
=================================================================
📖 RESULTS QUICK ACCESS GUIDE
=================================================================

📁 Results Directory Structure:
   results/
   ├── experiments/           # Individual experiment data
   │   ├── mnist/            # MNIST benchmark results
   │   ├── cifar10/          # CIFAR-10 results
   │   ├── nlp/              # NLP (IMDB) results
   │   ├── medical/          # Medical segmentation
   │   ├── resnet/           # ResNet18 results
   │   ├── highdim/          # High-dimensional functions
   │   ├── 2d_optimization/  # 2D test functions
   │   ├── robustness/       # Robustness analysis
   │   ├── sam_sensitivity/  # SAM sensitivity analysis
   │   ├── ablation/         # Basic ablation studies
   │   ├── advanced_ablation/       # 🆕 AMP, EMA, Label Smoothing
   │   ├── init_ablation/           # 🆕 Initialization studies
   │   ├── batch_ablation/          # Batch size ablation
   │   ├── lr_ablation/             # Learning rate ablation
   │   ├── wd_ablation/             # Weight decay ablation
   │   ├── scheduler_ablation/      # Scheduler ablation
   │   ├── optimizer_comparison/    # Statistical comparisons
   │   ├── hyperparam_sensitivity/  # 🆕 β, β1, β2 sweeps
   │   ├── convergence_validation/  # 🆕 Theory vs practice
   │   └── ablation_comprehensive/  # 🆕 Full ablations
   │
   ├── analysis/              # Statistical analyses
   │   ├── cross_experiment_aggregation.csv    # Combined results
   │   ├── optimizer_rankings.csv              # Overall rankings
   │   ├── cross_experiment_statistics.csv     # Statistical tests
   │   ├── 00_basic_statistics.csv             # Basic stats
   │   ├── 01_convergence_rates.csv            # Convergence analysis
   │   ├── 02_statistical_comparison.csv       # Pairwise comparisons
   │   └── dynamics_metrics/                   # 🆕 Dynamics analysis
   │
   ├── visualizations/        # Plots and visualizations
   │   ├── interactive/       # Interactive HTML plots
   │   ├── static/            # Static PNG/PDF plots
   │   ├── 2d_trajectories/   # 🆕 2D trajectory plots
   │   └── ablations/         # 🆕 Ablation visualizations
   │       ├── advanced_ablation/
   │       ├── init_ablation/
   │       └── ablation_comprehensive/
   │
   └── reports/               # Summary reports
       ├── experiment_summary_report.md
       └── 00_EXPERIMENT_SUMMARY.md

=================================================================

📊 Key Result Files:
   1. Cross-experiment aggregation:
      → analysis/cross_experiment_aggregation.csv
      
   2. Optimizer rankings (sorted by performance):
      → analysis/optimizer_rankings.csv
      
   3. Statistical significance tests:
      → analysis/cross_experiment_statistics.csv
      
   4. Convergence analysis:
      → analysis/01_convergence_rates.csv
   
   5. 🆕 Hyperparameter sensitivity:
      → experiments/hyperparam_sensitivity/momentum_beta_sweep*.csv
      → experiments/hyperparam_sensitivity/adam_beta_sweep*.csv
   
   6. 🆕 Convergence validation (Theory vs Practice):
      → experiments/convergence_validation/convergence_comparison.csv
      → experiments/convergence_validation/*_theory_vs_practice.png
   
   7. 🆕 Comprehensive ablations (with visualizations):
      → experiments/ablation_comprehensive/ablation_*.csv
      → experiments/ablation_comprehensive/*/visualizations/*.png
   
   8. 🆕 Training dynamics analysis:
      → analysis/dynamics_metrics/*_dynamics.csv
      → analysis/dynamics_metrics/*_dynamics.png

=================================================================

🔬 Statistical Analysis Features:
   ✅ Multi-seed experiments (10 seeds for high statistical power)
   ✅ Student's t-tests for significance
   ✅ Cohen's d effect sizes
   ✅ Statistical power analysis
   ✅ Multiple comparison corrections:
      - Holm-Bonferroni (FWER control)
      - Benjamini-Hochberg (FDR control)

=================================================================

🆕 NEW RESEARCH MODULES (Research Proposal Aligned):
   
   1. Hyperparameter Sensitivity Analysis
      • Momentum β sweeps: [0.0, 0.5, 0.9, 0.99]
      • Adam (β1, β2) configurations
      • Trajectory smoothness metrics
      • Oscillation index computation
      Location: experiments/hyperparam_sensitivity/
   
   2. Convergence Rate Validation
      • Theory vs practice O(1/k) validation
      • Curve fitting (linear/sublinear)
      • R² goodness-of-fit analysis
      • Theoretical bound overlays
      Location: experiments/convergence_validation/
   
   3. Comprehensive Ablation Studies
      • Momentum vs no-momentum
      • Adaptive vs fixed LR
      • Weight decay effects
      • With publication-quality visualizations
      Location: experiments/ablation_comprehensive/
   
   4. 2D Trajectory Visualization
      • Publication-quality contour plots
      • β effect visualization
      • Optimizer family comparisons
      Location: visualizations/2d_trajectories/
   
   5. Training Dynamics Analysis (NEW!)
      • Per-iteration gradient norms
      • Update magnitudes
      • Loss oscillations
      • Parameter space trajectories
      Location: analysis/dynamics_metrics/

=================================================================

🔄 Resume Logic:
   • Uses --resume flag to skip completed experiments
   • Checks for existing CSV files before running
   • Safe to interrupt and restart at any time
   • Progress is saved incrementally
   • ✅ All 22 experiments support resume

=================================================================

📈 Visualizations Available:
   • Training/test loss curves
   • Accuracy progression plots
   • Loss landscape 3D surfaces
   • Statistical comparison heatmaps
   • Convergence rate comparisons
   • Interactive parameter sensitivity plots
   • 🆕 2D optimizer trajectories with contours
   • 🆕 β parameter effect heatmaps
   • 🆕 Theory-practice comparison plots
   • 🆕 Ablation bar charts, box plots, heatmaps
   • 🆕 Training dynamics plots (gradient, update, oscillation)

=================================================================

💾 Download Results:
   Kaggle: Results archived to .zip in /kaggle/working/
           Download from Output tab after completion
   
   Local: Results saved to: {RESULTS_DIR}

=================================================================

📖 For detailed documentation, see:
   • README.md in results directory
   • reports/experiment_summary_report.md
   • Individual experiment folders for per-run details
   • RESEARCH_PROPOSAL_COVERAGE.md for academic alignment
   • CODEBASE_COMPLETENESS_REPORT.md for full audit
   • CRITICAL_GAPS_AND_FIXES.md for implementation details
   • FINAL_INTEGRATION_REPORT.md for completion status

=================================================================

🔬 Research Proposal Compliance:
   ✅ Theoretical convergence rate analysis
   ✅ Hyperparameter sensitivity (β, β1, β2)
   ✅ Trajectory dynamics visualization
   ✅ Theory vs practice validation
   ✅ Multi-seed statistical rigor
   ✅ Publication-quality outputs (300 DPI PNG + PDF)
   ✅ Non-convex test functions (Rosenbrock, Rastrigin, Ackley)
   ✅ Real neural network training dynamics

=================================================================
""".format(RESULTS_DIR=RESULTS_DIR))